In [1]:
# ══════════════════════════════════════════════════════════
# CELDA 1 — Librerías y carga de datos
# ══════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BASE    = '/content/drive/MyDrive/TFM_Seguridad_Vial'
outputs = f'{BASE}/outputs'

# Cargamos las dos tablas que necesitamos
df_anual   = pd.read_csv(f'{outputs}/tabla_anual_final.csv')
df_ipd     = pd.read_csv(f'{outputs}/IPD_completo.csv')

# Unimos IPD con tabla anual
df = df_anual.merge(
    df_ipd[['distrito', 'año', 'IPD', 'dim_seguridad', 'dim_vulnerabilidad']],
    on=['distrito', 'año'], how='left'
)

print(f'Forma: {df.shape}')
print(f'Años: {sorted(df["año"].unique())}')
print(f'Distritos: {df["distrito"].nunique()}')
print(f'Nulos: {df.isnull().sum().sum()}')

# Vista rápida de acc_ponderado por distrito y año
pivot = df.pivot_table(
    values='acc_ponderado',
    index='distrito',
    columns='año',
    aggfunc='sum'
)
print('\n=== ACC_PONDERADO POR DISTRITO Y AÑO ===')
print(pivot.to_string())

# Estadísticas básicas
print('\n=== ESTADÍSTICAS ACC_PONDERADO ===')
print(df['acc_ponderado'].describe().round(2))
print(f'\nVarianza entre distritos: {df.groupby("distrito")["acc_ponderado"].mean().var():.2f}')
print(f'Varianza entre años:      {df.groupby("año")["acc_ponderado"].mean().var():.2f}')

Mounted at /content/drive
Forma: (126, 29)
Años: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Distritos: 21
Nulos: 126

=== ACC_PONDERADO POR DISTRITO Y AÑO ===
año                  2019  2020  2021  2022  2023  2024
distrito                                               
ARGANZUELA            222   132   186   142   154   171
BARAJAS                52    37    60    56    70    55
CARABANCHEL           199   146   138   239   141   194
CENTRO                299   142   260   289   321   297
CHAMARTÍN             265   120   136   159   133   182
CHAMBERÍ              214   129   178   154   164   193
CIUDAD LINEAL         171   107   127   137   177   169
FUENCARRAL-EL PARDO   244   175   155   174   210   158
HORTALEZA             135   158   114   186   145   141
LATINA                190   181   174   171   225   170
MONCLOA-ARAVACA       132   159   143   180   188   168
MORATALAZ              69    49    51    67    39    65
PUE

In [2]:
# ¿En qué columnas están los nulos?
print(df.isnull().sum()[df.isnull().sum() > 0])

IPD                   42
dim_seguridad         42
dim_vulnerabilidad    42
dtype: int64


In [3]:
# ¿Cuánto cae acc_ponderado en 2020 respecto a 2019?
caida_covid = (
    df.groupby('año')['acc_ponderado'].mean()
    .reset_index()
    .rename(columns={'acc_ponderado': 'media'})
)
caida_covid['variacion_pct'] = caida_covid['media'].pct_change().round(3) * 100
print(caida_covid.to_string(index=False))

# ¿Es la caída uniforme entre distritos o hay variación?
df_2019 = df[df['año']==2019][['distrito','acc_ponderado']].rename(columns={'acc_ponderado':'ap_2019'})
df_2020 = df[df['año']==2020][['distrito','acc_ponderado']].rename(columns={'acc_ponderado':'ap_2020'})
covid = df_2019.merge(df_2020, on='distrito')
covid['caida_pct'] = ((covid['ap_2020'] - covid['ap_2019']) / covid['ap_2019'] * 100).round(1)
print('\n=== CAÍDA POR DISTRITO 2019→2020 ===')
print(covid.sort_values('caida_pct').to_string(index=False))

 año      media  variacion_pct
2019 170.619048            NaN
2020 133.904762          -21.5
2021 138.809524            3.7
2022 146.476190            5.5
2023 156.809524            7.1
2024 159.333333            1.6

=== CAÍDA POR DISTRITO 2019→2020 ===
           distrito  ap_2019  ap_2020  caida_pct
          CHAMARTÍN      265      120      -54.7
             CENTRO      299      142      -52.5
         ARGANZUELA      222      132      -40.5
           CHAMBERÍ      214      129      -39.7
      CIUDAD LINEAL      171      107      -37.4
          MORATALAZ       69       49      -29.0
            BARAJAS       52       37      -28.8
FUENCARRAL-EL PARDO      244      175      -28.3
        CARABANCHEL      199      146      -26.6
          SALAMANCA      258      193      -25.2
 PUENTE DE VALLECAS      222      171      -23.0
         VILLAVERDE      107       85      -20.6
              USERA      142      114      -19.7
  VILLA DE VALLECAS      109       88      -19.3
          